# ANATOMY ↔ GENE Relation-Wise Merge

Merges ANATOMY–GENE triples from multiple KG sources (DRKG, PrimeKG, Hetionet, TARKG),
aligns to a common schema, deduplicates by `(head, relation, tail)`, and saves the result.

## 0. Configuration

In [12]:
import os
import pandas as pd
import numpy as np

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/ANATOMY_CHEMICAL/ALL_ANATOMY_CHEMICAL.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Gene Lookup Dictionaries


In [13]:
UBERON = pd.read_csv(f'{BASE_DIR}data_collection/databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv')
UBERON_dict = dict(zip(UBERON['UBERON ID'], UBERON['Name']))
# UBERON_dict

## 2. Load Individual KG Sources

### 2a. Monarch

In [14]:
Monarch_Anatomy_chemical = pd.read_csv(f'{BASE_DIR}processed_data/Monarch/Monarch_final/Human/Human_Anatomy_Chemical.csv')
Monarch_Anatomy_chemical.columns = Monarch_Anatomy_chemical.columns.str.lower()
Monarch_Anatomy_chemical['head_detail_name'] = Monarch_Anatomy_chemical['head'].map(UBERON_dict)
print(f"DRKG:     {len(Monarch_Anatomy_chemical):,} rows | columns: {list(Monarch_Anatomy_chemical.columns)}")
Monarch_Anatomy_chemical['kg_type'] = 'Generalised'
Monarch_Anatomy_chemical['species'] = 'Homo sapiens'
Monarch_Anatomy_chemical['kg_source'] = 'Monarch_KG'

Monarch_Anatomy_chemical

DRKG:     9 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'relation_source', 'head_detail_name', 'tail_detail_name', 'tail_iupac_name', 'head_taxon_name', 'tail_taxon_name', 'head_id_is', 'tail_id_is', 'species_species', 'species']


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,tail_iupac_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,species_species,species,kg_type,kg_source
0,UBERON:0001752,AnatomicalEntity_ChemicalEntity,14781,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,enamel,hydroxylapatite,pentacalcium;hydroxide;triphosphate,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
1,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5280899,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,fovea centralis,zeaxanthin,"(1R)-4-[(1E,3E,5E,7E,9E,11E,13E,15E,17E)-18-[(...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
2,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5281243,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,fovea centralis,lutein,"(1R)-4-[(1E,3E,5E,7E,9E,11E,13E,15E,17E)-18-[(...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
3,UBERON:0001866,AnatomicalEntity_ChemicalEntity,638072,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,sebum,squalene,"(6E,10E,14E,18E)-2,6,10,15,19,23-hexamethyltet...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
4,UBERON:0001970,AnatomicalEntity_ChemicalEntity,5280352,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,bile,bilirubin IXalpha,3-[2-[[3-(2-carboxyethyl)-5-[(Z)-(3-ethenyl-4-...,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
5,UBERON:0002242,AnatomicalEntity_ChemicalEntity,446715,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,nucleus pulposus,keratan sulfate,"[(2R,3S,4S,5R,6R)-4-[(2S,3R,4R,5S,6R)-3-acetam...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
6,UBERON:0002280,AnatomicalEntity_ChemicalEntity,10112,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,otolith,calcium carbonate,calcium;carbonate,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
7,UBERON:0034948,AnatomicalEntity_ChemicalEntity,280,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,carbon dioxide in respiratory system,carbon dioxide,NaN,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG
8,UBERON:4000115,AnatomicalEntity_ChemicalEntity,14781,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,mineralized bone tissue,hydroxylapatite,pentacalcium;hydroxide;triphosphate,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens,Generalised,Monarch_KG


In [15]:
Monarch_Anatomy_chemical[Monarch_Anatomy_chemical['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,tail_iupac_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,species_species,species,kg_type,kg_source


## 3. Align Schemas and Concatenate

In [16]:
source_dfs = [
    Monarch_Anatomy_chemical,
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df['species'] = 'Homo sapiens'
final_df

Consolidated rows: 9


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,UBERON:0001752,AnatomicalEntity_ChemicalEntity,14781,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,enamel,hydroxylapatite,Generalised,Homo sapiens
1,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5280899,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,fovea centralis,zeaxanthin,Generalised,Homo sapiens
2,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5281243,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,fovea centralis,lutein,Generalised,Homo sapiens
3,UBERON:0001866,AnatomicalEntity_ChemicalEntity,638072,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,sebum,squalene,Generalised,Homo sapiens
4,UBERON:0001970,AnatomicalEntity_ChemicalEntity,5280352,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,bile,bilirubin IXalpha,Generalised,Homo sapiens
5,UBERON:0002242,AnatomicalEntity_ChemicalEntity,446715,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,nucleus pulposus,keratan sulfate,Generalised,Homo sapiens
6,UBERON:0002280,AnatomicalEntity_ChemicalEntity,10112,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,otolith,calcium carbonate,Generalised,Homo sapiens
7,UBERON:0034948,AnatomicalEntity_ChemicalEntity,280,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,carbon dioxide in respiratory system,carbon dioxide,Generalised,Homo sapiens
8,UBERON:4000115,AnatomicalEntity_ChemicalEntity,14781,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,mineralized bone tissue,hydroxylapatite,Generalised,Homo sapiens


## 4. QC — NaN Check Before Name Repair

In [17]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 6. Deduplicate and Add Schema Columns

Group by `(head, relation, tail)` and collapse duplicate `kg_source` values with `::` separator.

In [18]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})

# Enforce final column order
final_df_dedup = final_df_dedup[REQUIRED_COLS]

print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup.head()

After deduplication: 9 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,UBERON:0001752,AnatomicalEntity_ChemicalEntity,14781,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,enamel,hydroxylapatite,Generalised,Homo sapiens
1,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5280899,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,fovea centralis,zeaxanthin,Generalised,Homo sapiens
2,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5281243,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,fovea centralis,lutein,Generalised,Homo sapiens
3,UBERON:0001866,AnatomicalEntity_ChemicalEntity,638072,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,sebum,squalene,Generalised,Homo sapiens
4,UBERON:0001970,AnatomicalEntity_ChemicalEntity,5280352,AnatomicalEntity,related_to,ChemicalEntity,Monarch_KG,UBERON,Pubchem_ID,bile,bilirubin IXalpha,Generalised,Homo sapiens


In [19]:
print("\nkg_source values present:", set(final_df_dedup['kg_source']))


kg_source values present: {'Monarch_KG'}


In [20]:
print("\nkg_type values present:", set(final_df_dedup['kg_type']))


kg_type values present: {'Generalised'}


## 7. QC — NaN Check After Deduplication

In [21]:
nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 8. Save Output

In [22]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'


In [23]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 9 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/ANATOMY_CHEMICAL/ALL_ANATOMY_CHEMICAL.csv


In [24]:
#